# 07. Сравнение encoder и train-галерей после полного обогащения manual

MobileNetV3-Small (576D), MobileNetV3-Large V2 (960D), DINOv2 ViT-S/14 CLS (384D).
Frozen веса, L2 embeddings, одинаковый max-gallery-cosine score и глобальный отказ.
Размеры: 5/10/20/all независимых train-групп на POI; diverse/farthest-first и random SEED=42.
Все исходники остаются неизменными; source/author/licence сохранены.

Отдельные preprocessing encoder фиксируются в weights.json. DINOv2 использует 224 px для CPU:
это воспроизводимый стартовый вариант, а не максимально возможное качество DINOv2.
Пороги и галерея выбираются только по positive-val/negative-val при external FAR ≤1% на val.
Test после этого используется диагностически. Он уже участвовал в предыдущих экспериментах;
нужен новый закрытый полевой набор. Автоматической замены используемой модели нет.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
ROOT=Path.cwd().resolve()
if not (ROOT/'config/encoder_comparison.json').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.poi_baseline import sha256_file
OUTPUT=ROOT/'models/encoder_comparison_manual_all'
status=json.loads((OUTPUT/'status.json').read_text())
display(pd.Series(status,name='status'))
snapshot=json.loads((OUTPUT/'input_snapshot.json').read_text())
for name in ['train','val','test','negative_val','negative_test']:
    assert sha256_file(ROOT/f'data/annotations/{name}.csv')==snapshot[name],name
display(pd.read_csv(ROOT/'reports/manual_enrichment_plan.csv'))


mobilenetv3_small    complete
mobilenetv3_large    complete
dinov2_small         complete
Name: status, dtype: str

,poi_id,manual_folder,wikimedia_images,available_manual,selected_manual,selection_reason
0,moscow/ostankinskiy/main_entrance,Арка главного входа,46,24,24,all_manual
1,moscow/ostankinskiy/polymer_house,Дом Полимеров,0,12,12,all_manual
2,moscow/ostankinskiy/stone_flower,Каменный Цветок,0,23,23,all_manual
3,moscow/ostankinskiy/moskvarium,Москвариум,0,12,12,all_manual
4,moscow/ostankinskiy/ostankino_tower,Останкинская Телебашня,12,56,56,all_manual
5,moscow/ostankinskiy/armenia,Павильон Армения,42,113,113,all_manual
6,moscow/ostankinskiy/kazakhstan_pavilion,Павильон Казахстан,0,140,140,all_manual
7,moscow/ostankinskiy/cosmos,Павильон Космос,50,66,66,all_manual
8,moscow/ostankinskiy/moldova_pavilion,Павильон Молдова,0,42,42,all_manual
9,moscow/ostankinskiy/pavilion_5,Павильон №5,0,6,6,all_manual


## 1. Полнота сравнения и проверка артефактов

Если веса отсутствуют, соответствующий encoder остаётся missing_weights. Его метрики не выдумываются.
После появления официальных весов в .runtime/torch/checkpoints повторите scripts/compare_encoders.py:
завершённые encoder не меняются, недостающие продолжают общий протокол.


In [2]:
for name,state in status.items():
    if state!='complete':continue
    metadata=json.loads((OUTPUT/name/'metadata.json').read_text())
    for file,digest in metadata['artifact_sha256'].items():assert sha256_file(OUTPUT/name/file)==digest,file
    selection=json.loads((OUTPUT/name/'selection.json').read_text())
    assert not selection['test_used_for_selection']
    print(name,selection['variant'],'— hashes OK')
display(pd.read_csv(ROOT/'reports/encoder_comparison/comparison.csv'))


mobilenetv3_small diverse_all — hashes OK
mobilenetv3_large diverse_all — hashes OK
dinov2_small random_20 — hashes OK


,encoder,split,variant,query_count,positive_pairs,known_wrong_pairs,top1_accuracy,macro_top1_accuracy,true_accept_rate,false_reject_rate,...,wrong_accepts,hard_wrong_far,accepted_pair_precision,macro_true_accept_rate,open_set_far,external_unknown_tested,accuracy,macro_f1,external_any_poi_far,open_set_far_definition
0,mobilenetv3_small,val,diverse_all,191,191,4202,0.801047,0.740166,0.350785,0.649215,...,1,0.005236,0.985294,0.286192,0.008726,True,0.801047,0.736815,0.008726,any_poi_per_external_image
1,mobilenetv3_small,test,diverse_all,186,186,4092,0.822581,0.805757,0.349462,0.650538,...,1,0.005376,0.984848,0.273614,0.008726,True,0.822581,0.790916,0.008726,any_poi_per_external_image
2,mobilenetv3_large,val,diverse_all,191,191,4202,0.842932,0.782128,0.429319,0.570681,...,3,0.015707,0.964706,0.340872,0.008726,True,0.842932,0.771249,0.008726,any_poi_per_external_image
3,mobilenetv3_large,test,diverse_all,186,186,4092,0.801075,0.724561,0.387097,0.612903,...,2,0.010753,0.972973,0.348683,0.024433,True,0.801075,0.694768,0.024433,any_poi_per_external_image
4,dinov2_small,val,random_20,191,191,4202,0.905759,0.930661,0.481675,0.518325,...,1,0.005236,0.989247,0.500012,0.008726,True,0.905759,0.906567,0.008726,any_poi_per_external_image
5,dinov2_small,test,random_20,186,186,4092,0.892473,0.922532,0.467742,0.532258,...,2,0.010753,0.977528,0.481911,0.010471,True,0.892473,0.892196,0.010471,any_poi_per_external_image


## 2. Все варианты галерей: только validation

Оптимизация по нескольким вариантам делает val-результат оптимистичным. Не выбирайте вариант по test.
При all сохраняется один представитель независимой группы на POI; разные near-копии не считаются
дополнительными независимыми эталонами. Random детерминирован при SEED=42.


In [3]:
candidates=pd.read_csv(ROOT/'reports/encoder_comparison/validation_candidates.csv')
display(candidates[['encoder','variant','gallery_images','accuracy','macro_f1','true_accept_rate','external_any_poi_far']])
fig,ax=plt.subplots(figsize=(9,5))
caps=['5','10','20','all']
for name,group in candidates.groupby('encoder'):
    for method in ['diverse','random']:
        local=group[group.variant.str.startswith(method+'_')].copy()
        local['cap']=local.variant.str.split('_').str[-1].map({v:i for i,v in enumerate(caps)})
        local=local.sort_values('cap')
        ax.plot(local.cap,local.true_accept_rate*100,marker='o',linestyle='-' if method=='diverse' else '--',label=f'{name}/{method}')
ax.axhline(85,color='gray',linestyle=':',label='TAR target 85%')
ax.set(xlabel='Gallery cap per POI (independent train groups)',ylabel='Validation true accept, %',ylim=(0,100))
ax.set_xticks(range(4),caps);ax.legend(fontsize=8);fig.tight_layout()
fig.savefig(ROOT/'reports/encoder_comparison/validation_tradeoff.png',dpi=140)
plt.show()


,encoder,variant,gallery_images,accuracy,macro_f1,true_accept_rate,external_any_poi_far
0,mobilenetv3_small,diverse_5,114,0.607330,0.523880,0.036649,0.008726
1,mobilenetv3_small,random_5,114,0.575916,0.540668,0.151832,0.008726
2,mobilenetv3_small,diverse_10,220,0.659686,0.599382,0.136126,0.008726
3,mobilenetv3_small,random_10,220,0.612565,0.583697,0.209424,0.008726
4,mobilenetv3_small,diverse_20,404,0.727749,0.684931,0.256545,0.003490
5,mobilenetv3_small,random_20,404,0.701571,0.652981,0.277487,0.008726
6,mobilenetv3_small,diverse_all,883,0.801047,0.736815,0.350785,0.008726
7,mobilenetv3_large,diverse_5,114,0.617801,0.567810,0.162304,0.008726
8,mobilenetv3_large,random_5,114,0.623037,0.603236,0.188482,0.008726
9,mobilenetv3_large,diverse_10,220,0.654450,0.627362,0.214660,0.008726


C:\Users\Alisa\AppData\Local\Temp\ipykernel_8564\2780965648.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Per-POI диагностика выбранных по validation вариантов

Accuracy и macro-F1 оценивают распознавание без отказа; true accept — подтверждение правильной цели.
Их нельзя подменять друг другом. Строки test служат для отчёта; пороги после них не меняются.


In [4]:
for name,state in status.items():
    if state!='complete':continue
    print(name)
    display(pd.read_csv(OUTPUT/name/'test_per_poi.csv'))
display(Markdown((ROOT/'reports/encoder_comparison/summary.md').read_text(encoding='utf-8')))


mobilenetv3_small


,poi_id,query_count,top1_accuracy,true_accept_rate,known_wrong_far,true_accepts,wrong_accepts
0,moscow/arbat/arseny_morozov_house,7,0.857143,0.571429,0.000000,4,0
1,moscow/arbat/pashkov_house,7,0.857143,0.571429,0.000000,4,0
2,moscow/khamovniki/khrushchev_seleznev_house,5,1.000000,0.200000,0.000000,1,0
3,moscow/khamovniki/pushkin_museum,6,0.500000,0.166667,0.000000,1,0
4,moscow/ostankinskiy/armenia,23,0.913043,0.608696,0.000000,14,0
5,moscow/ostankinskiy/central_pavilion,16,0.812500,0.437500,0.000000,7,0
6,moscow/ostankinskiy/cosmos,17,0.823529,0.058824,0.000000,1,0
7,moscow/ostankinskiy/main_entrance,10,0.800000,0.300000,0.000000,3,0
8,moscow/ostankinskiy/moscow_sun,8,0.750000,0.250000,0.000000,2,0
9,moscow/ostankinskiy/ostankino_tower,10,1.000000,0.200000,0.000000,2,0


mobilenetv3_large


,poi_id,query_count,top1_accuracy,true_accept_rate,known_wrong_far,true_accepts,wrong_accepts
0,moscow/arbat/arseny_morozov_house,7,0.857143,0.428571,0.000000,3,0
1,moscow/arbat/pashkov_house,7,0.857143,0.714286,0.000000,5,0
2,moscow/khamovniki/khrushchev_seleznev_house,5,0.600000,0.200000,0.000000,1,0
3,moscow/khamovniki/pushkin_museum,6,0.666667,0.166667,0.000000,1,0
4,moscow/ostankinskiy/armenia,23,0.869565,0.652174,0.000000,15,0
5,moscow/ostankinskiy/central_pavilion,16,0.812500,0.187500,0.000000,3,0
6,moscow/ostankinskiy/cosmos,17,0.941176,0.176471,0.000000,3,0
7,moscow/ostankinskiy/main_entrance,10,0.900000,0.400000,0.000000,4,0
8,moscow/ostankinskiy/moscow_sun,8,0.875000,0.375000,0.000000,3,0
9,moscow/ostankinskiy/ostankino_tower,10,1.000000,0.100000,0.000000,1,0


dinov2_small


,poi_id,query_count,top1_accuracy,true_accept_rate,known_wrong_far,true_accepts,wrong_accepts
0,moscow/arbat/arseny_morozov_house,7,1.000000,1.000000,0.000000,7,0
1,moscow/arbat/pashkov_house,7,1.000000,0.571429,0.000000,4,0
2,moscow/khamovniki/khrushchev_seleznev_house,5,1.000000,0.200000,0.000000,1,0
3,moscow/khamovniki/pushkin_museum,6,1.000000,0.500000,0.000000,3,0
4,moscow/ostankinskiy/armenia,23,0.913043,0.434783,0.000000,10,0
5,moscow/ostankinskiy/central_pavilion,16,0.875000,0.000000,0.000000,0,0
6,moscow/ostankinskiy/cosmos,17,0.764706,0.470588,0.000000,8,0
7,moscow/ostankinskiy/main_entrance,10,1.000000,0.000000,0.000000,0,0
8,moscow/ostankinskiy/moscow_sun,8,0.875000,0.750000,0.000000,6,0
9,moscow/ostankinskiy/ostankino_tower,10,1.000000,0.800000,0.000000,8,0


# Сравнение frozen encoder и галерей

Все encoder используют один обогащённый датасет и train-only галереи. Выбор размера/способа галереи и глобальных порогов — только по val/negative-val.
Размеры галерей: 5, 10, 20 и все независимые train-группы на POI; diverse/farthest-first и random (SEED=42). Score=max cosine, без Ridge и геометрии.
Каждый encoder использует свой документированный preprocessing, crop 224. DINOv2 CLS 384D — eval-only реализация совместимой архитектуры, загрузка официальных весов strict=True; 224 px выбран для CPU, это не максимум качества DINOv2.
Изменения данных не сравниваются напрямую с прежними версиями: честное сравнение encoder — внутри этого выпуска.
Текущий test диагностический: он уже использовался в предыдущих экспериментах. Нужен новый закрытый полевой test. Перебор галерей может давать оптимистичную val-оценку.

| Encoder | Статус |
|---|---|
| mobilenetv3_small | complete |
| mobilenetv3_large | complete |
| dinov2_small | complete |

| Encoder | Split | Gallery | Accuracy | Macro-F1 | True accept | External FAR |
|---|---|---|---:|---:|---:|---:|
| mobilenetv3_small | val | diverse_all | 80.10% | 73.68% | 35.08% | 0.87% |
| mobilenetv3_small | test | diverse_all | 82.26% | 79.09% | 34.95% | 0.87% |
| mobilenetv3_large | val | diverse_all | 84.29% | 77.12% | 42.93% | 0.87% |
| mobilenetv3_large | test | diverse_all | 80.11% | 69.48% | 38.71% | 2.44% |
| dinov2_small | val | random_20 | 90.58% | 90.66% | 48.17% | 0.87% |
| dinov2_small | test | random_20 | 89.25% | 89.22% | 46.77% | 1.05% |

Предпочтительный encoder по validation: dinov2_small, random_20. Автоматической замены основной модели нет.
На диагностическом test выбранного варианта: true accept 46.77%, external FAR 1.05%. Цель 85% true accept не достигнута.
Ограничение external FAR ≤1% на test не выполнено; вариант остаётся экспериментальным.

## Наблюдение → интерпретация → действие

- Малые классы и зависимые ракурсы → метрики нестабильны → смотреть per-POI таблицы и добирать независимые мобильные кадры.
- После полного добавления manual меняются размер и баланс классов → общий прирост не доказывает улучшение на мобильных фото → смотреть by_source.csv; состав POI у источников различается, поэтому это диагностика, а не причинное сравнение источников.
- Калибровка ограничивает эмпирический external FAR на val → это не гарантия FAR на новых объектах → оценивать отдельный test и не подстраивать по нему пороги.
- Source/author/licence сохранены в snapshots и каждой галерее → происхождение прослеживается → сохранять атрибуцию при последующем использовании.
- mobilenetv3_large: test FAR 2.44% >1% → validation-порог не удержал ограничение на других негативных POI → не менять его по test, сохранить вариант экспериментальным.
- dinov2_small: test FAR 1.05% >1% → validation-порог не удержал ограничение на других негативных POI → не менять его по test, сохранить вариант экспериментальным.
- dinov2_small: максимальная val accuracy 96.34% у diverse_all, true accept 35.60%; выбран random_20 с true accept 48.17% → ранжирование и подтверждение цели дают разный выбор галереи → оценивать true accept совместно с FAR.


## 4. Повторение

Из корня проекта: `python scripts/compare_encoders.py`, затем этот Notebook → Run All.
Поддерживается `--extra-site .venv/Lib/site-packages` для рабочего Python.
Новый состав данных или конфигурация требуют нового --output. Чтобы открыть другой выпуск,
измените OUTPUT и путь reports в ноутбуке. Новые веса автоматически не скачиваются.

Источники архитектур и весов:
- https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_large.html
- https://github.com/facebookresearch/dinov2

Eval-only DINOv2 реализован через стандартные PyTorch-операции с именами параметров upstream;
официальный checkpoint загружается strict=True. Дальнейшие оптимизации должны проверяться
на совпадение с официальной реализацией перед эксплуатацией.
